In [1]:
#%pip install requests beautifulsoup4
import pandas as pd
import requests
from bs4 import BeautifulSoup
url = "https://quotes.toscrape.com/"

In [2]:
url = "http://books.toscrape.com/"
data = {'titre du livre': [], 'prix': [], 'disponibilité': []}
contenu_site = requests.get(url)
extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
descrip = extract_contenu.find('ol', class_='row')
#print(descrip)
for element in descrip.find_all('li', class_='col-xs-6 col-sm-4 col-md-3 col-lg-3'):
    titre = element.find('h3').find('a').text
    prix = element.find("div", class_="product_price").find("p", class_="price_color").text
    dispo = element.find("div", class_="product_price").find("p", class_="instock availability").text.strip()
    # .strip() pour enlever les espaces et les sauts de ligne
    data['titre du livre'].append(titre)
    data['prix'].append(prix)
    data['disponibilité'].append(dispo)
data = pd.DataFrame(data)
data.head()

,titre du livre,prix,disponibilité
0,A Light in the ...,Â£51.77,In stock
1,Tipping the Velvet,Â£53.74,In stock
2,Soumission,Â£50.10,In stock
3,Sharp Objects,Â£47.82,In stock
4,Sapiens: A Brief History ...,Â£54.23,In stock


In [3]:
url = "https://quotes.toscrape.com/"
contenu_site = requests.get(url)
extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
data = []
bloc_citation = extract_contenu.find_all('div', class_='quote') 

for citation in bloc_citation:
    text = citation.find('span', class_='text').get_text()
    author = citation.find('small', class_='author').get_text()
    tags = [tag.get_text() for tag in citation.find_all('a', class_='tag')]
    tags_f = ""
    for tag in tags:
        tags_f = tags_f +""+ tag + ","
        
    data.append([text, author, tags_f])
data = pd.DataFrame(data, columns=['citation', 'auteur', 'tags'])
    
data_csv = data.to_csv('citations.csv', index=False)
data.shape

(10, 3)

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Dans cette je récupère les citations de la première page et je les stocke dans une liste
url = "https://quotes.toscrape.com/"
contenu_site = requests.get(url)
extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
data = []

# Première page
bloc_citation = extract_contenu.find_all('div', class_='quote') 
for citation in bloc_citation:
    texte = citation.find('span', class_='text').get_text()
    auteur = citation.find('small', class_='author').get_text()
    tags = [tag.get_text() for tag in citation.find_all('a', class_='tag')]
    tags_final = ", ".join(tags)  # Simplification
    data.append([texte, auteur, tags_final])

# Récupération de la première page suivante
next_link = extract_contenu.find('li', class_='next')
# Si la page suivante existe, on ajoute son lien à la liste next_page
next_page = [next_link.find("a")["href"] if next_link else None]

k = 0
# Tant que la page suivante existe, on continue à extraire les données
while isinstance(next_page[k], str) and next_page[k]:
    # Correction de l'URL
    url = f"https://quotes.toscrape.com/{next_page[k]}"
    #print(f"url suivant : {url}")            
    # Récupération du contenu de la page suivante
    contenu_site = requests.get(url)
    # Extraction du contenu de la page suivante
    extract_contenu = BeautifulSoup(contenu_site.text, 'html.parser') 
    # Extraction de toutes les citations de la page suivante
    bloc_citation = extract_contenu.find_all('div', class_='quote') 
    
    for citation in bloc_citation:
        texte = citation.find('span', class_='text').get_text()
        auteur = citation.find('small', class_='author').get_text()
        tags = [tag.get_text() for tag in citation.find_all('a', class_='tag')]
        tags_final = ", ".join(tags)
        data.append([texte, auteur, tags_final])
    
    # Vérification de la page suivante
    next_link = extract_contenu.find('li', class_='next')
    if next_link and next_link.find("a"):
        next_page.append(next_link.find("a")["href"])
    else:
        print("Aucune page suivante trouvée. Fin de l'extraction.")
        break
    
    k = k + 1

# Création du DataFrame
data = pd.DataFrame(data, columns=['citation', 'auteur', 'tags'])
data_csv = data.to_csv('citations.csv', index=False)
data.head(100)



Aucune page suivante trouvée. Fin de l'extraction.


,citation,auteur,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
...,...,...,...
95,“You never really understand a person until yo...,Harper Lee,better-life-empathy
96,“You have to write the book that wants to be w...,Madeleine L'Engle,"books, children, difficult, grown-ups, write, ..."
97,“Never tell the truth to people who are not wo...,Mark Twain,truth
98,"“A person's a person, no matter how small.”",Dr. Seuss,inspirational
